# EAZY broadband photo-$z$
$$p(z \mid x) \propto \exp[-\chi^2(z)/2]p(z)$$
Run EAZY-py on the same LSST-like broadband photometry as `broadband_flow.ipynb`
and `bootstrapped_flow.ipynb`: CFHT u + HSC grizy with degraded uncertainties.
Fit the **full photometric test split**, then evaluate only its secure-spec subset.
No spectroscopic redshift or COSMOS PDF is supplied to EAZY, and no additional
noise is drawn. This is an untuned template baseline, not the full-multiband
COSMOS2020 EAZY catalog.

Use the project Conda kernel. Restore `data/parent.npz`, the EAZY resources and
the COSMOS EBV extract listed in [the data notes](../bin/data/README.md#optional-eazy-broadband-baseline).
Templates/filters are from [eazy-photoz](https://github.com/gbrammer/eazy-photoz/tree/f651e94d91042a84906090a66d2a3fec75504e21).
The cells below expose data preparation, fitting, PDFs, evaluation and saving,
in the same sequence as the two flow notebooks.

In [ ]:
from pathlib import Path
import hashlib
import json
import time

import eazy
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table

from pzflow.utils import PDFSampler, coverage

In [ ]:
seed = 20261116
r_prior = False  # Flat in z by default; True enables the supplied approximate R prior.
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'setup.py').exists())
parent_path = root / 'data/parent.npz'
assets = root / 'data/eazy-photoz-master'
ebv_path = root / 'data/cosmos2020_specz/classic_i25_ebv.csv'
magnitude_path = root / 'data/cosmos2020_specz/classic_magnitude_intervals.csv'
stem = 'eazy_rprior' if r_prior else 'eazy'
work_dir = root / 'results' / stem / 'notebook'
work_dir.mkdir(parents=True, exist_ok=True)

# Resolve the local resource directory before importing EAZY's fitting classes.
if not (assets / 'filters/FILTER.RES.latest').exists() or not ebv_path.exists():
    raise FileNotFoundError('Restore EAZY resources and the EBV extract; see bin/data/README.md')
eazy.set_data_path(str(assets))
from eazy.photoz import PhotoZ
from eazy.filters import FilterFile

In [ ]:
plt.rcParams.update(
    {
        'font.family': 'serif',
        'font.size': 11,
        'axes.linewidth': 1.5,
        'xtick.major.size': 5,
        'ytick.major.size': 5,
        'xtick.major.width': 1.5,
        'ytick.major.width': 1.5,
        'legend.frameon': False,
    }
)

## 2. Load the data and preserve the spatial split
Fluxes and errors are in microJy, ordered u, g, r, i, z, y with shape (N, 6).
Use the saved fixed degraded realization, as in the flow test predictions.
The spatial split is 0 = training, 1 = validation, 2 = test. EAZY uses fixed
public templates; it does not train on either training split or test spectroscopy.

In [ ]:
keys = ['id', 'flux', 'fluxerr', 'zspec', 'has_specz', 'split', 'imag', 'cell']
with np.load(parent_path) as f:
    if not json.loads(str(f['metadata'])).get('canonical_parent'):
        raise ValueError('Use the canonical parent from match_cosmos.py')
    data = {key: f[key] for key in keys}

assert data['flux'].shape == data['fluxerr'].shape == (len(data['id']), 6)
assert np.isfinite(data['flux']).all()
assert np.isfinite(data['fluxerr']).all() and (data['fluxerr'] > 0).all()
assert len(np.unique(data['id'])) == len(data['id'])
for a, b in [(0, 1), (0, 2), (1, 2)]:
    assert not np.intersect1d(
        data['cell'][data['split'] == a], data['cell'][data['split'] == b]
    ).size
phot_test = np.flatnonzero(data['split'] == 2)
ids = data['id'][phot_test]
flux = data['flux'][phot_test].astype(float).copy()
err = data['fluxerr'][phot_test].astype(float).copy()
print(f'Fit {len(phot_test):,} photometric test galaxies')

## 3. Prepare the broadband fluxes for template fitting
Correct for Galactic extinction using each object's released `ebv_mw`, matched
by Classic ID. Multiply flux **and** error by the same factor, preserving SNR
and negative measurements. EAZY receives native CFHT/HSC filter curves, not Rubin
curves. These deterministic corrections match `bin/run_eazy.py`.

With `r_prior=True`, also apply the common aperture-to-total factor `total_off2`
to every band and error. This defines the total HSC r magnitude for EAZY's supplied
R prior, an approximate R-to-HSC-r mapping. With a flat prior the free SED amplitude
absorbs a common aperture scale, so no total correction is needed.

In [ ]:
def match_rows(path, ids):
    # Auxiliary extracts can have a different selection and ordering from the parent.
    table = np.atleast_1d(np.genfromtxt(path, delimiter=',', names=True))
    table = table[np.argsort(table['id'])]
    j = np.searchsorted(table['id'], ids)
    if len(np.unique(table['id'])) != len(table) or np.any(j == len(table)):
        raise ValueError(f'Missing or duplicate IDs in {path}')
    if not np.array_equal(table['id'][j], ids):
        raise ValueError(f'Missing IDs in {path}')
    return table[j]


ebv = match_rows(ebv_path, ids)['ebv_mw']
assert np.isfinite(ebv).all()
coeff = np.array([4.807, 3.69, 2.715, 2.0, 1.515, 1.298])
correction = 10 ** (0.4 * ebv[:, None] * coeff[None, :])
flux *= correction
err *= correction
if r_prior:
    total_off = match_rows(magnitude_path, ids)['total_off2']
    factor = 10 ** (-0.4 * total_off)
    assert np.isfinite(factor).all() and (factor > 0).all()
    flux *= factor[:, None]
    err *= factor[:, None]
assert np.isfinite(flux).all() and np.isfinite(err).all() and (err > 0).all()
np.testing.assert_allclose(
    flux / err, data['flux'][phot_test] / data['fluxerr'][phot_test], rtol=1e-6
)

## 4. Configure the template baseline
Use the same fixed 12 tweaked FSPS templates, template-error amplitude 0.20,
1% systematic-error floor and redshift grid as the existing EAZY script.
No zero-point learning, template optimization, beta prior or fixed spec-z.
The default is flat in redshift; the optional R prior changes the saved filename
so both runs can coexist. All configuration is chosen before inspecting test labels.

In [ ]:
filter_ids = [409, 314, 315, 316, 317, 318]  # CFHT cfh9302 u; HSC g/r/i/z/y.
filters = FilterFile(str(assets / 'filters/FILTER.RES.latest'))
catalog = Table({'id': ids})
for k, fid in enumerate(filter_ids):
    catalog[f'F{fid}'] = flux[:, k]
    catalog[f'E{fid}'] = err[:, k]
catalog_path = work_dir / 'catalog.fits'
catalog.write(catalog_path, overwrite=True)
translate = work_dir / 'eazy.translate'
translate.write_text('')  # F409/E409-style names already identify the filters.

params = dict(
    CATALOG_FILE=str(catalog_path),
    CATALOG_FORMAT='fits',
    FILTERS_RES=str(assets / 'filters/FILTER.RES.latest'),
    TEMPLATES_FILE=str(assets / 'templates/fsps_full/tweak_fsps_QSF_12_v3.param'),
    TEMP_ERR_FILE=str(assets / 'templates/uvista_nmf/template_error_10.def'),
    WAVELENGTH_FILE=str(assets / 'templates/uvista_nmf/lambda.def'),
    PRIOR_FILE=str(assets / 'templates/prior_F160W_TAO.dat'),
    OUTPUT_DIRECTORY=str(work_dir),
    MAIN_OUTPUT_FILE=stem,
    PRIOR_ABZP=23.9,
    APPLY_PRIOR='n',
    FIX_ZSPEC='n',
    Z_MIN=0.001,
    Z_MAX=10.0,
    Z_STEP=0.005,
    MW_EBV=0.0,
    CAT_HAS_EXTCORR='y',
    SYS_ERR=0.01,
    TEMP_ERR_A2=0.20,
    VERBOSITY=0,
    ADD_CGM='n',
    NOT_OBS_THRESHOLD=-1.0e10,
)
if r_prior:
    # EAZY-py needs a commented header for this original C-code prior file.
    prior_file = work_dir / 'prior_R_zmax7_py.dat'
    prior_file.write_text('# ' + (assets / 'templates/prior_R_zmax7.dat').read_text())
    params.update(PRIOR_FILE=str(prior_file), PRIOR_FILTER=315, APPLY_PRIOR='y')

## 5. Fit the photometric test split
EAZY fits nonnegative template amplitudes at each redshift, then constructs
p(z) with the chosen prior. The catalog contains IDs, fluxes and errors only.
`n_proc=0` uses one process; the full default run fits every photometric test row.

In [ ]:
start = time.monotonic()
p = PhotoZ(
    params=params,
    translate_file=str(translate),
    load_prior=r_prior,
    n_proc=0,
    random_seed=seed,
)
assert p.NOBJ == len(ids) and p.NFILT == 6 and np.all(p.nusefilt == 6)
np.testing.assert_array_equal(np.asarray(p.cat['id']), ids)
p.fit_catalog(n_proc=0, get_best_fit=False, prior=r_prior, beta_prior=False)
p.compute_lnp(prior=r_prior, beta_prior=False, clip_wavelength=None)
p.evaluate_zml(prior=r_prior, beta_prior=False, clip_wavelength=None)
fit_seconds = time.monotonic() - start
print(f'Fit completed in {fit_seconds:.1f} s')

## 6. Normalize PDFs and predict redshift quantiles
Treat EAZY's saved grid values as a piecewise-linear density in redshift.
Normalize by its trapezoidal integral, then invert the integrated density for
2.5%, 16%, 50%, 84% and 97.5% quantiles, matching the flow output convention.
The posterior median used in comparisons is distinct from EAZY's `zml`.

In [ ]:
def pdf_quantiles(sampler, probs=(0.025, 0.16, 0.5, 0.84, 0.975)):
    rows = np.arange(len(sampler.p))
    quantiles = []
    for prob in probs:
        j = np.clip((sampler.c <= prob).sum(axis=1) - 1, 0, len(sampler.grid) - 2)
        width = np.diff(sampler.grid)[j]
        a = sampler.p[rows, j]
        slope = (sampler.p[rows, j + 1] - a) / width
        mass = np.maximum(prob - sampler.c[rows, j], 0)
        den = a + np.sqrt(np.maximum(a * a + 2 * slope * mass, 0))
        step = np.divide(2 * mass, den, out=np.zeros_like(mass), where=den > 0)
        quantiles.append(sampler.grid[j] + np.clip(step, 0, width))
    return np.column_stack(quantiles)


grid = np.asarray(p.zgrid)
pdf = np.exp(p.lnp - p.lnp.max(axis=1)[:, None])
assert np.isfinite(pdf).all() and (pdf >= 0).all()
norm = np.trapezoid(pdf, grid, axis=1)
assert (norm > 0).all()
pdf /= norm[:, None]
np.testing.assert_allclose(np.trapezoid(pdf, grid, axis=1), 1, atol=1e-6)
sampler = PDFSampler(grid, pdf)
q_all = pdf_quantiles(sampler)
assert q_all.shape == (len(phot_test), 5)
assert np.isfinite(q_all).all() and (np.diff(q_all, axis=1) >= 0).all()
print(f'Predicted {len(q_all):,} test galaxies without redshift labels')

## 7. Evaluate where independent test spectroscopy is available
Use exactly the flow notebooks' `has_specz` subset of the same photometric test
split. Compute median normalized bias, NMAD, outlier fraction and central interval
coverage. These describe the selected spectroscopic sample, not the complete
photometric population. Template choices are not tuned using these diagnostics.

In [ ]:
scored = data['has_specz'][phot_test]
test = phot_test[scored]
ztest = data['zspec'][test]
q = q_all[scored]
assert np.isfinite(ztest).all() and (ztest > 0).all()
pit = sampler.pit(ztest, indices=np.flatnonzero(scored))

# Evaluate the same piecewise-linear PDF for NLL; floor only the log at float tiny.
rows = np.flatnonzero(scored)
j = np.clip(np.searchsorted(grid, ztest, side='right') - 1, 0, len(grid) - 2)
w = (ztest - grid[j]) / np.diff(grid)[j]
density = sampler.p[rows, j] * (1 - w) + sampler.p[rows, j + 1] * w
density[(ztest < grid[0]) | (ztest > grid[-1])] = 0
nll = -np.log(np.maximum(density, np.finfo(float).tiny))
residual = (q[:, 2] - ztest) / (1 + ztest)
c68, c95 = coverage(pit)
metrics = {
    'n': len(test),
    'bias': float(np.median(residual)),
    'nmad': float(1.4826 * np.median(np.abs(residual - np.median(residual)))),
    'outlier_fraction': float(np.mean(np.abs(residual) > 0.15)),
    'coverage68': float(c68),
    'coverage95': float(c95),
    'median_width68': float(np.median(q[:, 3] - q[:, 1])),
}
print(json.dumps(metrics, indent=2))

In [ ]:
fig = plt.figure(figsize=(6, 6))
sub = fig.add_subplot(111)
sub.hexbin(ztest, q[:, 2], gridsize=60, bins='log', mincnt=1, cmap='viridis')
sub.plot([0, 5], [0, 5], 'k--', lw=1)
sub.set(xlim=(0, 5), ylim=(0, 5), xlabel=r'$z_{\mathrm{spec}}$', ylabel=r'$z_{\mathrm{median}}$')
plt.show()

In [ ]:
credibility = np.linspace(0, 1, 101)
rank = np.abs(2 * pit - 1)
curve = np.array([(rank <= level).mean() for level in credibility])
fig = plt.figure(figsize=(6, 6))
sub = fig.add_subplot(111)
sub.plot(credibility, curve, label='EAZY')
sub.plot([0, 1], [0, 1], 'k--', lw=1, label='Nominal')
sub.set(xlim=(0, 1), ylim=(0, 1), xlabel='Nominal credibility', ylabel='Test coverage')
sub.legend()
plt.show()

Example PDFs span the same test sample's native i-band magnitudes.
Dashed lines show held-out spectroscopy for evaluation only.

In [ ]:
order = np.argsort(data['imag'][test])
examples = order[np.linspace(0, len(order) - 1, 6, dtype=int)]
fig, axes = plt.subplots(2, 3, figsize=(10, 5.5), constrained_layout=True)
for ax, j in zip(axes.flat, examples):
    row = rows[j]
    ax.plot(grid, pdf[row], label='EAZY PDF')
    ax.axvline(ztest[j], color='k', ls='--', label=r'$z_{\mathrm{spec}}$')
    ax.set(xlim=(0, min(10, max(q[j, 4], ztest[j]) + 0.3)), xlabel=r'$z$', ylabel=r'$p(z)$')
    ax.set_title(f'ID {data["id"][test[j]]}; i = {data["imag"][test[j]]:.2f}')
axes.flat[0].legend(fontsize=8)
plt.show()

## 8. Save PDFs and predictions
Use the same unprefixed `quantiles`, `pit`, `nll` and parent identity fields as
the flow notebooks. `data/eazy_test_predictions.npz` can be added to the `inputs`
and `models` dictionaries in `comparison.ipynb`. The optional R-prior run uses
`eazy_rprior` instead. Also save all photometric test quantiles and full PDFs for
reuse, including deterministic corrected inputs and EAZY `zml`.

All generated files are local and ignored. Re-running replaces this notebook's
selected EAZY outputs; it does not change flow outputs. The manifest records
sample counts, settings and hashes of the parent, auxiliary catalogs and actual
template/filter resources. No checkpoint training is required.

In [ ]:
prediction = {key: data[key][test] for key in ['id', 'zspec', 'imag', 'flux', 'fluxerr']}
prediction.update(quantiles=q, pit=pit, nll=nll)
np.savez_compressed(root / f'data/{stem}_test_predictions.npz', **prediction)
np.savez_compressed(
    root / f'data/{stem}_photometric_test_predictions.npz',
    id=ids,
    flux=data['flux'][phot_test],
    fluxerr=data['fluxerr'][phot_test],
    quantiles=q_all,
)
np.savez_compressed(
    root / f'data/{stem}_posteriors.npz',
    id=ids,
    grid=grid,
    pdf=pdf,
    zml=p.zml,
    flux=data['flux'][phot_test],
    fluxerr=data['fluxerr'][phot_test],
    eazy_flux=flux,
    eazy_fluxerr=err,
)

resources = {
    'parent': parent_path,
    'ebv': ebv_path,
    'filters': assets / 'filters/FILTER.RES.latest',
    'template_list': Path(params['TEMPLATES_FILE']),
    'template_error': Path(params['TEMP_ERR_FILE']),
    'wavelength': Path(params['WAVELENGTH_FILE']),
}
for line in resources['template_list'].read_text().splitlines():
    fields = line.split()
    if fields and not fields[0].startswith('#'):
        resources['template_' + fields[0]] = assets / fields[1]
if r_prior:
    resources.update(magnitudes=magnitude_path, prior=assets / 'templates/prior_R_zmax7.dat')
inputs = {}
for name, path in resources.items():
    with path.open('rb') as f:
        inputs[name] = {'path': str(path), 'sha256': hashlib.file_digest(f, 'sha256').hexdigest()}
manifest = {
    'code': 'EAZY-py',
    'version': eazy.__version__,
    'numpy_version': np.__version__,
    'seed': seed,
    'sample': 'full photometric split 2; evaluate has_specz subset',
    'n_photometric_test': len(phot_test),
    'n_spec_test': len(test),
    'n_templates': p.NTEMP,
    'fit_seconds': fit_seconds,
    'params': params,
    'parent_sha256': inputs['parent']['sha256'],
    'inputs': inputs,
    'metrics': metrics,
    'filters': [
        {'band': b, 'filter_id': fid, 'name': filters[fid].name}
        for b, fid in zip('ugrizy', filter_ids)
    ],
    'prior': 'Supplied R prior on total HSC r; approximate mapping; no beta prior'
    if r_prior
    else 'Flat in redshift; no magnitude or beta prior',
    'corrections': 'Released EBV coefficients [4.807,3.69,2.715,2.0,1.515,1.298]; '
    'flux and error scaled together; total_off2 only for R prior',
    'limitations': [
        'Untuned templates and zero points; selected spectroscopic evaluation',
        'Native CFHT/HSC responses with LSST-like noise, not Rubin bandpasses',
    ],
    'nll_log_floor': float(np.finfo(float).tiny),
}
with (root / f'data/{stem}_run.json').open('w') as f:
    json.dump(manifest, f, indent=2, allow_nan=False)
    f.write(chr(10))
print(f'Saved {stem} predictions and full PDFs under data/')